# PravahX â€” Delft3D Flexible Mesh (D-Flow FM) Remote Runner

This notebook restores your pre-compiled Delft3D FM binaries from Google Drive, optionally installs the minimal Intel runtime, executes a prepared case bundle from PravahX, and securely saves the cryptographically verified results back to Google Drive.


In [ ]:
%%python
import os

from google.colab import drive

drive.mount("/content/drive")
print("\n\u2713 Google Drive mounted.")

In [ ]:
%%bash
set -e
BUNDLE_PATH="/content/drive/MyDrive/PravahX/delft3dfm_linux_x86_64.tar.gz"
SHA_PATH="${BUNDLE_PATH}.sha256"

if [ ! -f "$BUNDLE_PATH" ]; then
    echo "âœ— Bundle not found at $BUNDLE_PATH"
    exit 1
fi

if [ -f "$SHA_PATH" ]; then
    echo "Verifying SHA256 checksum..."
    EXPECTED_SHA=$(cat "$SHA_PATH" | awk '{print $1}')
    KNOWN_GOOD_SHA="491364fbba88948752356fb1af0edc8aaecc496ba95545a9b66494c0ba1a6258"
    echo "Read from Drive .sha256: $EXPECTED_SHA"
    echo "Known good benchmark SHA:  $KNOWN_GOOD_SHA"
    ACTUAL_SHA=$(sha256sum "$BUNDLE_PATH" | awk '{print $1}')
    if [ "$EXPECTED_SHA" != "$ACTUAL_SHA" ]; then
        echo "âœ— Checksum mismatch! Expected: $EXPECTED_SHA, Actual: $ACTUAL_SHA"
        exit 1
    else
        echo "âœ“ Checksum verified."
    fi
else
    echo "No .sha256 file found for bundle; skipping verification."
fi

echo "Extracting bundle..."
tar -xzf "$BUNDLE_PATH" -C /content
echo "âœ“ Bundle extracted to /content/delft3d_bin."

echo "Checking dependencies for dflowfm..."
export LD_LIBRARY_PATH=/content/delft3d_bin/lib:$LD_LIBRARY_PATH
MISSING=$(ldd /content/delft3d_bin/bin/dflowfm | grep "not found" || true)

if [ -n "$MISSING" ]; then
    echo "Missing libraries detected:"
    echo "$MISSING"
    echo "Installing Intel oneAPI runtime..."
    wget -qO- https://apt.repos.intel.com/intel-gpg-keys/GPG-PUB-KEY-INTEL-SW-PRODUCTS.PUB | gpg --dearmor | sudo tee /usr/share/keyrings/oneapi-archive-keyring.gpg > /dev/null
    echo "deb [signed-by=/usr/share/keyrings/oneapi-archive-keyring.gpg] https://apt.repos.intel.com/oneapi all main" | sudo tee /etc/apt/sources.list.d/oneAPI.list
    sudo apt-get update -qq
    sudo apt-get install -y -qq intel-oneapi-runtime-fortran intel-oneapi-runtime-mpi
    export LD_LIBRARY_PATH=/opt/intel/oneapi/compiler/latest/linux/compiler/lib/intel64_lin:/opt/intel/oneapi/mpi/latest/lib:$LD_LIBRARY_PATH
    echo "Re-checking dependencies..."
    ldd /content/delft3d_bin/bin/dflowfm | grep "not found" || echo "âœ“ All dependencies satisfied."
else
    echo "âœ“ All dependencies satisfied."
fi

echo "=== Final Check: dflowfm version ==="
/content/delft3d_bin/bin/dflowfm --version || (echo "âœ— dflowfm version check failed" && exit 1)
echo "âœ“ Delft3D FM is ready for simulation."


In [ ]:
%%python
import tarfile

from google.colab import files

print("Please upload the *_linux_package.tar.gz file generated by PravahX's remote mode.")
uploaded = files.upload()
os.makedirs("/content/case_run", exist_ok=True)
for fn in uploaded:
    print(f"Extracting {fn}...")
    with tarfile.open(fn, "r:gz") as tar:
        try:
            tar.extractall("/content/case_run", filter="data")
        except TypeError:
            tar.extractall("/content/case_run")
    print("\u2713 Extracted to /content/case_run.")

In [ ]:
%%bash
cd /content/case_run
if [ ! -f "run_linux.sh" ]; then
    echo "âœ— run_linux.sh not found in the uploaded package."
    exit 1
fi

chmod +x run_linux.sh
echo "=== Starting Simulation ==="
./run_linux.sh
EXIT_CODE=$?

if [ $EXIT_CODE -ne 0 ]; then
    echo "âœ— Simulation script exited with code $EXIT_CODE"
    exit $EXIT_CODE
fi

echo "Checking .dia log files for errors..."
DIA_FILES=$(find . -name "*.dia")
if [ -n "$DIA_FILES" ]; then
    if grep -q -E "ERROR|FATAL" $DIA_FILES; then
        echo "âœ— Errors found in .dia logs:"
        grep -E "ERROR|FATAL" $DIA_FILES
        exit 1
    fi
fi

TIMESTAMP=$(date -u +%Y%m%dT%H%M%SZ)
MDU_NAME=$(ls *.mdu 2>/dev/null | head -n 1 || echo "case")
CASE_BASE=${MDU_NAME%.mdu}
ARCHIVE_NAME="delft3d_results_${CASE_BASE}_${TIMESTAMP}.tar.gz"

echo "Packaging results to $ARCHIVE_NAME..."
tar -czvf "/content/$ARCHIVE_NAME" DFM_OUTPUT_*
cp "/content/$ARCHIVE_NAME" "/content/drive/MyDrive/PravahX/$ARCHIVE_NAME"
echo "âœ“ Results packaged and securely saved to Google Drive: MyDrive/PravahX/$ARCHIVE_NAME"
echo "Download this archive and point PravahX to it in import mode."
